## 1. Setup Notebook

**Model configuration**
- Select `.yaml` file defining model, solver, and outputs  
- All configs are located in `configs/`
- To beginn three models for three different PD read-outs of Dupilumab are available.
    - `EASI`, `TARC`, `RO` 
    - Read Docstring in `models/...py` for details
    - Add new models by registrating in the `MODEL_TO_CONFIG` dictionary

**Run modes**
- `"run_missing"`: run only simulations without existing results  
- `"overwrite"`: rerun all simulations (overwrite existing results)  
- `"load_only"`: load existing summaries only (no simulations executed)  

**Parallelization (`workers`)**
- Number of simulations executed in parallel (separate processes)  
- Guideline: use ~60–80% of available CPU cores  
  - Example: 12-core machine → 8–10 workers is typically optimal
 
**Repository setup**
- Detects and sets the repository root path  
- Adds repo root to `sys.path` for module imports  
- Prints: `.../PK-PD-Automation-main`

In [ ]:
# -------------------------
# INPUT (edit only here)
# -------------------------

# Model selection: (must exist in MODEL_TO_CONFIG below)
MODEL = "MODEL_NAME"

# Run modes:
MODE = "run_missing"

# Parallelization: Number of parallel workers
workers = 8  

# -------------------------
# IMPORTS
# -------------------------

import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, PercentFormatter, LogFormatterSciNotation
from matplotlib.gridspec import GridSpec
from scipy.interpolate import PchipInterpolator
import importlib, hashlib, json as _json

# -------------------------
# Resolve repository root
# -------------------------

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "scripts" / "runner.py").exists()
            and (candidate / "configs").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate repository root. Please run this notebook from inside the repo "
        "or check that scripts/runner.py, configs/, and models/ exist."
    )

repo_root = find_repo_root()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
from scripts.runner import run_sweep, load_config

# -------------------------
#  REGISTER NOVEL MODEL HERE
# -------------------------

# Maps model names to their YAML configuration files
MODEL_TO_CONFIG = {
    "EASI": repo_root / "configs" / "EASI_sweep.yaml",
    "TARC": repo_root / "configs" / "TARC_sweep.yaml",
    "RO": repo_root / "configs" / "RO_sweep.yaml"
}

# Resolve selected config
config_path = MODEL_TO_CONFIG[MODEL]
print("Using config:", config_path)

## 2. Define parameter grid + run sweep

### Input variables

- **`DOSE_MODE`**  
Select one of these two mutually exclusive input modes:

    - `manual`
      - Provide an explicit list of dose values  
      - Example:
        ```python
        dose_values_manual = [0.1, 1, 10, 100]
        ```

    - `geometric`
      - Automatically generate log-spaced doses  
      - Format: `[lower, upper, n]`
        - `lower`: minimum dose (> 0)
        - `upper`: maximum dose
        - `n`: number of points  
      - Example:
        ```python
        dose_specification = [0.1, 100, 20]
        ```

- **`interval_values`**  
  Enter list of all interval durations wished for grid generation (in weeks)

- **`param_overrides`**  
  Dictionary of parameter values that will temporarily overwrite the YAML/model defaults.  
  **Note**: "Varibale name" has to be equal to name in `.yaml` file  
  ```python
  param_overrides = {"Cl_ml_day": 45}

### Execution

- Generates full simulation grid:  
  `len(dose_values) × len(interval_values)`

- Calls `Runner.py` to run or load simulations (depending on `MODE`)

- Automatically:
  - Stores results under a parameter-specific folder (`params_<hash>`)
  - Loads all available summaries
  - Filters results strictly to the requested grid

### Validation / completeness check

- Compares:
  - **Expected runs** (grid size)
  - **Actual runs** (found summaries)

- If incomplete:
  - Prints missing `(dose, interval)` combinations

→ Ensures sweep integrity before plotting or analysis

In [ ]:
# -------------------------
# INPUT (edit only here)
# -------------------------


# "geometric": generate log-spaced doses [lower, upper, n]
dose_values_geometric = [0.01, 500, 30]


# "manual": provide explicit list
dose_values_manual = [0.1, 1, 10, 100]


# Choose dose input:
DOSE_MODE = "geometric"


# Dosing intervals (weeks): provide explicit list with integers only
interval_values = [2, 4, 8, 12, 24]


# Parameter overrides
param_overrides = { }


# -------------------------
# GEOMETRIC SERIES FUNCTION
# -------------------------

def geometric_series(lower, upper, n, decimals=2): 
    if lower <= 0 or upper <= 0: 
        raise ValueError("Bounds must be > 0 for a geometric series.") 
    if n < 2: 
        raise ValueError("n must be at least 2.") 
    r = (upper / lower) ** (1 / (n - 1)) 
    vals = [round(lower * r**i, decimals) for i in range(n)] 
    vals = sorted(set(vals)) 
    return vals

# -------------------------
# RESOLVE DOSE VALUES
# -------------------------

if DOSE_MODE == "manual":
    if not dose_values_manual:
        raise ValueError("DOSE_MODE='manual' but dose_values_manual is empty.")
    dose_values = sorted(set(dose_values_manual))

elif DOSE_MODE == "geometric":
    dose_values = geometric_series(*dose_values_geometric, decimals=2)

else:
    raise ValueError(f"Unknown DOSE_MODE: {DOSE_MODE}")


print("Dose values:", dose_values)
print("Intervals:", interval_values)


# -------------------------
# CONFIG + PARAMETER HASHING
# -------------------------

# Load model config and module
cfg = load_config(config_path)
model_module = importlib.import_module(cfg["model"]["module"])


def canonicalize_params(obj):
    """
    Convert parameters into a deterministic, JSON-stable format
    (ensures reproducible hashing).
    """
    if isinstance(obj, dict):
        return {k: canonicalize_params(obj[k]) for k in sorted(obj)}
    if isinstance(obj, (list, tuple)):
        return [canonicalize_params(x) for x in obj]
    if hasattr(obj, "item") and callable(obj.item):
        try:
            return obj.item()
        except Exception:
            pass
    if isinstance(obj, float):
        return float(f"{obj:.12g}")
    return obj


def params_fingerprint(params: dict) -> str:
    """
    Generate unique hash for a parameter set.
    Used to separate results for different model configurations.
    """
    payload = _json.dumps(canonicalize_params(params), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


# Build effective parameter set:
# DEFAULTS + YAML params + overrides
effective_params = dict(getattr(model_module, "DEFAULTS", {}))
effective_params.update(cfg.get("params", {}))
if param_overrides:
    effective_params.update(param_overrides)

# Compute parameter ID → determines output folder
param_id = params_fingerprint(effective_params)

base_root = repo_root / cfg["output"]["root_dir"]
out_root = base_root / f"params_{param_id}"

print("Loading/running in:", out_root)


# -------------------------
# LOAD HELPERS
# -------------------------

def load_all_summaries(out_root: Path) -> pd.DataFrame:
    """
    Load all run_summary.json files from parameter folder.
    """
    rows = []
    for p in sorted(out_root.glob("*/run_summary.json")):
        with open(p, "r", encoding="utf-8") as f:
            d = json.load(f)
        d["_run_dir"] = str(p.parent)
        rows.append(d)
    return pd.DataFrame(rows)


def filter_to_grid_exact(df: pd.DataFrame, dose_values, interval_values) -> pd.DataFrame:
    """
    Keep only rows matching the requested (dose, interval) grid.
    Prevents mixing results from previous runs.
    """
    if df.empty:
        return df

    df = df.copy()
    df["dose_mgkg"] = df["dose_mgkg"].astype(float)
    df["interval_weeks"] = df["interval_weeks"].astype(int)

    requested = set((float(d), int(i)) for d in dose_values for i in interval_values)

    mask = [
        (float(r.dose_mgkg), int(r.interval_weeks)) in requested
        for r in df.itertuples(index=False)
    ]
    return df.loc[mask]


# -------------------------
# RUN OR LOAD
# -------------------------

if MODE == "load_only":
    # Only read existing results
    df_all = load_all_summaries(out_root)

else:
    # Run sweep (parallelized via `workers`)
    df = run_sweep(
        config_path=config_path,
        dose_values=dose_values,
        interval_values=interval_values,
        workers=workers,
        overwrite=(MODE == "overwrite"),
        param_overrides=param_overrides,
    )

    # Reload all summaries (ensures consistency)
    df_all = load_all_summaries(out_root)


# Filter strictly to requested grid
df = filter_to_grid_exact(df_all, dose_values, interval_values)

# Sort for clean downstream plotting
if not df.empty:
    df["interval_weeks"] = df["interval_weeks"].astype(int)
    df = df.sort_values(["interval_weeks", "dose_mgkg"]).reset_index(drop=True)


# -------------------------
# COMPLETENESS CHECK
# -------------------------

expected = len(dose_values) * len(interval_values)
actual = len(df)

print("Expected runs:", expected)
print("Found runs:", actual)

# Identify missing simulations
if actual != expected:
    found = set(
        (float(r.dose_mgkg), int(r.interval_weeks))
        for r in df.itertuples(index=False)
    )

    missing = [
        (float(d), int(i))
        for d in dose_values
        for i in interval_values
        if (float(d), int(i)) not in found
    ]

    print("Missing combos:", missing[:30], "..." if len(missing) > 30 else "")

## 3. Plot sweep output

This cell generates exposure-response plot from the current sweep results in `df`.

### Output
- **Left panel:** PD response vs. `C_avg`
- **Right panel:** PD response vs. `C_trough`

### User settings

- **PD masking**
  - `PD_MIN` and `PD_MAX` define which PD values are displayed
  - Purpose: Keep plot clean by removing unnecessary data points around 0 and Emax

- **Axis settings**
  - `Y_AXIS_TITLE` to set a custom y-axis label (uses pd_key name if empty)
  - `XAUTO = True` automatically chooses an appropriate x-axis range from the plotted data
  - `YAUTO = True` automatically chooses an appropriate y-axis range from the plotted data
  - If `XAUTO` or `YAUTO` is `False`, the manual settings are used instead

- **Summary text**
  - `show_summary_text = True` adds a text block below the plot with:
    - model parameters
    - PD masking settings
    - shown dose ranges after masking


In [ ]:
# -------------------------
# INPUT (edit only here)
# -------------------------

# PD masking
PD_MIN = 0.01
PD_MAX = 0.99


# X-axis settings
XAUTO = True
XMIN = 0.01
XMAX = 1000
XTICKS = [0.01, 0.1, 1, 10, 100, 1000]


# Y-axis settings
Y_AXIS_TITLE = "Y-AXIS TITLE"
YAUTO = True
YMIN = 0
YMAX = 100


# Summary text
show_summary_text = True

# -------------------------
# CHECK INPUT DATA
# -------------------------

if df.empty:
    raise RuntimeError("DataFrame `df` is empty. Run the sweep or load summaries first.")

# -------------------------
# RESOLVE PD COLUMN + Y LABEL
# -------------------------

pd_key = cfg.get("outputs", {}).get("pd_key")
if not pd_key:
    raise RuntimeError(
        "Could not determine PD output key. Please define outputs.pd_key in the config."
    )

pd_col = f"{pd_key}_trough"
if pd_col not in df.columns:
    raise RuntimeError(
        f"Expected PD trough column '{pd_col}' not found in df. "
        f"Available columns: {df.columns.tolist()}"
    )

y_title = Y_AXIS_TITLE.strip() if Y_AXIS_TITLE.strip() else pd_key

# -------------------------
# SETUP COLORS AND INTERVALS
# -------------------------

intervals = sorted(df["interval_weeks"].astype(int).unique())
base_colors = ["#FF8C00", "#FF0040", "#00B7EB", "#006400", "#7A1FA2", "#D3B683"]

interval_color = {
    iw: base_colors[idx % len(base_colors)]
    for idx, iw in enumerate(intervals)
}

# -------------------------
# HELPER FUNCTIONS
# -------------------------

def smooth_curve_logx(x, y, n_points=250):
    """
    Smooth y over log10(x) using PCHIP interpolation.
    Returns smoothed x and y arrays, or (None, None) if not enough points.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = (x > 0) & np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return None, None

    x = x[mask]
    y = y[mask]

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    xu, inv = np.unique(x, return_inverse=True)
    if len(xu) != len(x):
        y_sum = np.zeros_like(xu, dtype=float)
        counts = np.zeros_like(xu, dtype=int)
        for i, idx in enumerate(inv):
            y_sum[idx] += y[i]
            counts[idx] += 1
        x = xu
        y = y_sum / counts

    if len(x) < 2:
        return None, None

    lx = np.log10(x)
    lx_smooth = np.linspace(lx.min(), lx.max(), n_points)
    y_smooth = PchipInterpolator(lx, y)(lx_smooth)
    x_smooth = 10 ** lx_smooth
    return x_smooth, y_smooth

def choose_log_ticks(xmin, xmax):
    """
    Generate decade ticks for a log-scaled x-axis.
    """
    if xmin <= 0 or xmax <= 0:
        raise ValueError("Log-scale x-axis requires positive xmin and xmax.")

    lo = int(np.floor(np.log10(xmin)))
    hi = int(np.ceil(np.log10(xmax)))
    ticks = [10 ** k for k in range(lo, hi + 1)]

    if xmin not in ticks:
        ticks.insert(0, float(xmin))
    if xmax not in ticks:
        ticks.append(float(xmax))

    return sorted(set(float(t) for t in ticks))

def style_axis(ax, xmin, xmax, ymin, ymax, xticks, has_any=True):
    """
    Apply common formatting to one panel.
    """
    ax.set_xscale("log")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{t:g}" for t in xticks])
    ax.xaxis.set_major_formatter(LogFormatterSciNotation())
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
    ax.grid(which="both", linestyle=":", linewidth=0.7, alpha=0.7)

    if not has_any:
        ax.text(
            0.5, 0.5,
            "No points after masking",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=12, alpha=0.7,
        )

def extract_xy(dfi, x_col, pd_col, pd_min, pd_max):
    """
    Return filtered x and y(%) arrays for one interval and one x variable.
    """
    x = dfi[x_col].astype(float).to_numpy()
    y = dfi[pd_col].astype(float).to_numpy()

    mask = (
        np.isfinite(x) & (x > 0) &
        np.isfinite(y) & (y >= pd_min) & (y <= pd_max)
    )

    return x[mask], 100 * y[mask]

def auto_x_limits(plot_data):
    """
    Determine automatic log-scale x-limits across all panels.
    """
    xvals = []
    for item in plot_data:
        xvals.extend(item["C_avg_ugml"][0])
        xvals.extend(item["C_trough_ugml"][0])

    if not xvals:
        raise RuntimeError("No positive finite x-values remain after masking. Cannot determine automatic x-axis.")

    xvals = np.asarray(xvals, dtype=float)
    xmin = float(10 ** np.floor(np.log10(np.min(xvals))))
    xmax = float(10 ** np.ceil(np.log10(np.max(xvals))))
    return xmin, xmax

def auto_y_limits(plot_data):
    """
    Determine automatic y-limits in percent across all panels.
    """
    yvals = []
    for item in plot_data:
        yvals.extend(item["C_avg_ugml"][1])
        yvals.extend(item["C_trough_ugml"][1])

    if not yvals:
        raise RuntimeError("No PD values remain after masking. Cannot determine automatic y-axis.")

    yvals = np.asarray(yvals, dtype=float)
    ymin = float(np.min(yvals))
    ymax = float(np.max(yvals))

    yr = ymax - ymin
    if yr <= 0:
        yr = max(1.0, 0.05 * ymax if ymax != 0 else 1.0)

    pad = 0.05 * yr
    ymin = max(0.0, ymin - pad)
    ymax = min(100.0, ymax + pad)

    if ymax <= ymin:
        ymin = max(0.0, ymin - 1.0)
        ymax = min(100.0, ymax + 1.0)

    return ymin, ymax

def draw_panel(ax, plot_data, x_key, xlabel, xmin, xmax, ymin, ymax, xticks):
    """
    Draw one exposure-response panel.
    """
    any_points = False

    for item in plot_data:
        x, y = item[x_key]
        color = item["color"]

        in_range = (x >= xmin) & (x <= xmax)
        x = x[in_range]
        y = y[in_range]

        if len(x) == 0:
            continue

        any_points = True

        xs, ys = smooth_curve_logx(x, y)
        if xs is not None:
            ax.plot(xs, ys, color=color, linewidth=2.0, alpha=0.95)

        ax.scatter(
            x, y,
            color=color,
            edgecolor="k",
            linewidth=0.6,
            s=45,
            zorder=3,
        )

    style_axis(ax, xmin, xmax, ymin, ymax, xticks, has_any=any_points)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(f"{y_title} (%)", fontsize=12)

def format_param_summary(cfg, model_module, effective_params):
    """
    Return text block with model parameters and optional microconstants.
    """
    lines = [f"Model: {cfg['model']['name']}", "Parameters:"]

    for k, v in sorted(effective_params.items()):
        lines.append(f"  {k} = {v:g}" if isinstance(v, float) else f"  {k} = {v}")

    if hasattr(model_module, "microconstants"):
        try:
            mcs = model_module.microconstants(effective_params)
            if mcs:
                lines.append("Microconstants:")
                for k, v in sorted(mcs.items()):
                    lines.append(f"  {k} = {v:g}" if isinstance(v, float) else f"  {k} = {v}")
        except Exception as e:
            lines.append(f"Microconstants: error ({e})")

    return "\n".join(lines)

def format_plot_settings(PD_MIN, PD_MAX, XAUTO, YAUTO, XMIN, XMAX, YMIN, YMAX, y_title):
    """
    Return text block summarizing masking and axis settings.
    """
    lines = [
        "Plot settings:",
        f"  PD_MIN   = {PD_MIN:g}",
        f"  PD_MAX   = {PD_MAX:g}",
        f"  Y_LABEL  = {y_title}",
        f"  XAUTO    = {XAUTO}",
        f"  YAUTO    = {YAUTO}",
    ]

    if not XAUTO:
        lines += [f"  XMIN     = {XMIN:g}", f"  XMAX     = {XMAX:g}"]
    if not YAUTO:
        lines += [f"  YMIN     = {YMIN:g}", f"  YMAX     = {YMAX:g}"]

    return "\n".join(lines)

def format_dose_ranges_shown(df, intervals, pd_col, mask_min=None, mask_max=None, decimals=5):
    """
    Return shown dose ranges after PD masking for each interval.
    """
    lines = ["Shown dose ranges (after masking):"]

    for iw in intervals:
        dfi = df[df["interval_weeks"].astype(int) == int(iw)].copy()

        if mask_min is not None:
            dfi = dfi[dfi[pd_col].astype(float) >= mask_min]
        if mask_max is not None:
            dfi = dfi[dfi[pd_col].astype(float) <= mask_max]

        if dfi.empty:
            lines.append(f"  q{iw}w: (none)")
            continue

        doses = dfi["dose_mgkg"].astype(float).to_numpy()
        lines.append(f"  q{iw}w: {doses.min():.{decimals}f}–{doses.max():.{decimals}f} mg/kg")

    return "\n".join(lines)

# -------------------------
# PREPARE PLOTTED DATA
# -------------------------

plot_data = []

for iw in intervals:
    dfi = df[df["interval_weeks"].astype(int) == int(iw)].copy()

    plot_data.append({
        "interval": iw,
        "color": interval_color[iw],
        "C_avg_ugml": extract_xy(dfi, "C_avg_ugml", pd_col, PD_MIN, PD_MAX),
        "C_trough_ugml": extract_xy(dfi, "C_trough_ugml", pd_col, PD_MIN, PD_MAX),
    })

# -------------------------
# RESOLVE AXIS LIMITS
# -------------------------

if XAUTO:
    active_xmin, active_xmax = auto_x_limits(plot_data)
    active_xticks = choose_log_ticks(active_xmin, active_xmax)
else:
    active_xmin = float(XMIN)
    active_xmax = float(XMAX)
    active_xticks = list(XTICKS)

if YAUTO:
    active_ymin, active_ymax = auto_y_limits(plot_data)
else:
    active_ymin = float(YMIN)
    active_ymax = float(YMAX)

# -------------------------
# CREATE FIGURE
# -------------------------

if show_summary_text:
    fig = plt.figure(figsize=(12, 7))
    gs = GridSpec(2, 2, height_ratios=[3.2, 1.6], hspace=0.28, wspace=0.22)

    ax_cavg = fig.add_subplot(gs[0, 0])
    ax_ctr = fig.add_subplot(gs[0, 1])
    ax_txt = fig.add_subplot(gs[1, :])
    ax_txt.axis("off")
else:
    fig, (ax_cavg, ax_ctr) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
    ax_txt = None

# -------------------------
# DRAW PLOTS
# -------------------------

draw_panel(
    ax=ax_cavg,
    plot_data=plot_data,
    x_key="C_avg_ugml",
    xlabel=r"$C_{\mathrm{avg}}$ (µg/mL)",
    xmin=active_xmin,
    xmax=active_xmax,
    ymin=active_ymin,
    ymax=active_ymax,
    xticks=active_xticks,
)

draw_panel(
    ax=ax_ctr,
    plot_data=plot_data,
    x_key="C_trough_ugml",
    xlabel=r"$C_{\mathrm{trough}}$ (µg/mL)",
    xmin=active_xmin,
    xmax=active_xmax,
    ymin=active_ymin,
    ymax=active_ymax,
    xticks=active_xticks,
)

# -------------------------
# LEGEND
# -------------------------

legend_handles = [
    plt.Line2D([0], [0], color=interval_color[iw], lw=2, marker="o", markersize=6, label=f"q{iw}w")
    for iw in intervals
]

legend_y = 0.99 if show_summary_text else 1.02
fig.legend(
    handles=legend_handles,
    title="Interval",
    fontsize=12,
    loc="upper center",
    ncol=max(1, len(intervals)),
    bbox_to_anchor=(0.5, legend_y),
)


# -------------------------
# SUMMARY TEXT PANEL
# -------------------------

if show_summary_text:
    summary_text = format_param_summary(
        cfg=cfg,
        model_module=model_module,
        effective_params=effective_params,
    )

    settings_text = format_plot_settings(
        PD_MIN=PD_MIN,
        PD_MAX=PD_MAX,
        XAUTO=XAUTO,
        YAUTO=YAUTO,
        XMIN=active_xmin if XAUTO else XMIN,
        XMAX=active_xmax if XAUTO else XMAX,
        YMIN=active_ymin if YAUTO else YMIN,
        YMAX=active_ymax if YAUTO else YMAX,
        y_title=y_title,
    )

    ranges_text = format_dose_ranges_shown(
        df=df,
        intervals=intervals,
        pd_col=pd_col,
        mask_min=PD_MIN,
        mask_max=PD_MAX,
        decimals=5,
    )

    ax_txt.text(
        0.01, 0.98,
        summary_text + "\n\n" + settings_text + "\n\n" + ranges_text,
        ha="left",
        va="top",
        fontsize=11,
        family="monospace",
        transform=ax_txt.transAxes,
    )

# -------------------------
# LAYOUT + SHOW FIGURE
# -------------------------

plt.tight_layout(rect=[0, 0, 1, 0.8])

latest_fig = fig
plt.show()

## 4. Save most recent plot

This cell saves the most recently generated figure from cell 3.

### Behavior
- Saves `latest_fig` as a `.png`
- Save location:
  `.../results/sweep_plots/<MODEL>`
- Existing files with the same name are overwritten

### File name
The exported file name includes:
- model name
- dose range
- interval range

### Requirement
- Cell 3 must be run first so that `latest_fig` exists

In [ ]:
# -------------------------
# SAVE SETTINGS
# -------------------------

# Resolution
save_dpi = 100

# -------------------------
# BASIC CHECK
# -------------------------

if "latest_fig" not in globals():
    raise RuntimeError("No plot found. Run cell 3 first to generate `latest_fig`.")

if latest_fig is None:
    raise RuntimeError("`latest_fig` is None. Run cell 3 first.")


# -------------------------
# HELPER
# -------------------------

def sanitize_for_filename(value):
    """
    Convert numeric value into a compact filename-safe string.
    Example: 0.01 -> 0p01
    """
    value = float(value)
    return f"{value:g}".replace(".", "p")


# -------------------------
# SAVE PATH
# -------------------------

plot_dir = repo_root / "results" / "sweep_plots" / MODEL
plot_dir.mkdir(parents=True, exist_ok=True)


# -------------------------
# BUILD FILE NAME
# -------------------------

dose_min = sanitize_for_filename(df["dose_mgkg"].astype(float).min())
dose_max = sanitize_for_filename(df["dose_mgkg"].astype(float).max())
interval_min = int(df["interval_weeks"].astype(int).min())
interval_max = int(df["interval_weeks"].astype(int).max())

png_name = (
    f"{MODEL}"
    f"_dose_{dose_min}_to_{dose_max}"
    f"_interval_{interval_min}_to_{interval_max}.png"
)

png_path = plot_dir / png_name


# -------------------------
# SAVE FIGURE
# -------------------------

# Overwrites existing file with same name
latest_fig.savefig(png_path, dpi=save_dpi, bbox_inches="tight")

print("Saved plot to:", png_path)